# 00 — Setup and data (M0)

**Goal:** confirm the environment works and get the raw QM9 files onto disk, verified.

M0 acceptance criteria (PLAN §9.4):
- the stack is installed with pinned versions (`requirements.txt`), and `pytest` runs;
- the QM9 tarball, `uncharacterized.txt` and `readme.txt` are in `data/raw/`, with checksums logged in `docs/DATA.md`.

Run with the **Python (qff26)** kernel. Every cell is safe to re-run.

## 1. Environment

These versions are recorded with every results file (CLAUDE.md conventions).

In [1]:
import pandas as pd

from qm9dipole import REPO_ROOT
from qm9dipole.provenance import git_hash, package_versions

print("repo:", REPO_ROOT)
print("git :", git_hash())
pd.Series(package_versions(), name="version").to_frame()

repo: C:\Users\dougl\OneDrive\Documents\dev_projects\Quantum_hackathon\QML_project
git : nocommit


,version
python,3.12.15
qiskit,2.5.2
qiskit-aer,0.17.2
qiskit-ibm-runtime,0.50.0
numpy,2.5.3
scipy,1.18.1
scikit-learn,1.9.1
pandas,3.0.6
pyarrow,25.0.1
matplotlib,3.11.2


In [2]:
import subprocess
import sys

r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no"], cwd=REPO_ROOT, capture_output=True, text=True)
print(r.stdout[-2000:])
assert r.returncode == 0, "test suite failed"

.......                                                                  [100%]
7 passed in 1.11s



## 2. Download

Three files from the figshare collection behind Ramakrishnan et al. (2014). Each download is checked
against the size and MD5 that figshare publishes, and we also record a SHA-256 for `docs/DATA.md`.

The tarball holds 133,885 tiny `.xyz` files. We **never extract it**: this folder is synced by OneDrive,
and 133k small files would swamp the sync. Python's `tarfile` reads members straight out of the
compressed stream instead.

In [3]:
from qm9dipole.data import RAW_DIR, fetch_all

manifest = pd.DataFrame(fetch_all())
manifest

,file,figshare_id,bytes,md5,sha256
0,dsgdb9nsd.xyz.tar.bz2,3195389,86144227,ad1ebd51ee7f5b3a6e32e974e5d54012,3a63848ac80691bdb8d41834b575afad345b9300d7a2db...
1,uncharacterized.txt,3195404,486752,a361887bacb427b8a0ce7903d92a53b4,3aa5115d540b356de94791d4a74c3bf1ed91c469ecf52a...
2,readme.txt,3195392,5198,c6581a03f673746528c57acfc6b79679,0ee83fc21faa9527ebe89fad0ce8f2bd2da1cbd35a3c7e...


## 3. Sanity checks on the raw files

In [4]:
import tarfile

from qm9dipole.data import TARBALL

n_members, first = 0, None
with tarfile.open(TARBALL, "r:bz2") as tar:  # decompresses as it streams; nothing is written to disk
    for member in tar:
        n_members += 1
        if first is None:
            first = tar.extractfile(member).read().decode()

print(f"{n_members:,} molecule files in the tarball\n")
assert n_members == 133_885
print(first)

133,885 molecule files in the tarball

5
gdb 1	157.7118	157.70997	157.70699	0.	13.21	-0.3877	0.1171	0.5048	35.3641	0.044749	-40.47893	-40.476062	-40.475117	-40.498597	6.469	
C	-0.0126981359	 1.0858041578	 0.0080009958	-0.535689
H	 0.002150416	-0.0060313176	 0.0019761204	 0.133921
H	 1.0117308433	 1.4637511618	 0.0002765748	 0.133922
H	-0.540815069	 1.4475266138	-0.8766437152	 0.133923
H	-0.5238136345	 1.4379326443	 0.9063972942	 0.133923
1341.307	1341.3284	1341.365	1562.6731	1562.7453	3038.3205	3151.6034	3151.6788	3151.7078
C	C	
InChI=1S/CH4/h1H4	InChI=1S/CH4/h1H4



That is molecule 1, **methane** (CH₄). How to read it (readme, "Format"):

| Line | Content |
|---|---|
| 1 | number of atoms (5) |
| 2 | `gdb 1` then 15 properties; the 4th after the index is **μ in debye** — here `0.` |
| 3–7 | element, x, y, z (Å), **Mulliken charge** |
| then | vibrational frequencies, SMILES (GDB / relaxed), InChI |

- μ = 0 for methane is what symmetry predicts: four identical C–H bond dipoles point to the corners of a
  tetrahedron and cancel, just like CO₂ in PLAN §1.1.
- The 5th atom column (Mulliken charge) comes from the same DFT calculation as μ. Using it as a feature would
  leak the answer, so it is **never** an input (CLAUDE.md rule 1).
- Line 2 starts `gdb 1` with a *space*, then tabs. The M1 parser must split on any whitespace.

In [5]:
uncharacterized_ids = [
    int(tok[0])
    for line in (RAW_DIR / "uncharacterized.txt").read_text().splitlines()
    if (tok := line.split()) and tok[0].isdigit()
]
print(f"{len(uncharacterized_ids):,} uncharacterized molecules, e.g. ids {uncharacterized_ids[:5]}")
assert len(uncharacterized_ids) == 3054

3,054 uncharacterized molecules, e.g. ids [58, 61, 80, 185, 186]


These 3,054 failed the authors' consistency check: the relaxed DFT geometry corresponds to a different
molecule (different SMILES) than the one GDB-17 specified. PLAN §3.4 excludes them by default.

The readme also flags a few hard-to-converge molecules. Whether to exclude those is an M1 decision:

In [6]:
readme = (RAW_DIR / "readme.txt").read_text()
print(readme[readme.index("Notes"):readme.index("References")].strip())

Notes
-----

Out of the 133885 molecules, geometries of the 11 molecules with indices 
21725, 87037, 59827, 117523, 128113, 129053, 129152, 129158, 130535, 6620, 59818 
were difficult to converge.
Low threshold convergence was possible for 21725, 59827, 128113, 129053, 129152, 130535.
Molecules 6620 and 59818 converged to very low-lying saddlepoints, with lowest frequency < 10i cm^-1.


## M0 status

If every cell above ran without an assertion error, M0 is complete:
- environment and versions are recorded above; pins are in `requirements.txt`;
- the three raw files are verified against figshare's MD5s, with checksums logged in `docs/DATA.md`.

**Next (M1):** parse the tarball into `data/processed/qm9.parquet`, apply and document exclusions, and build the
nested splits.